In [28]:
%load_ext autoreload
%autoreload 2
from dig4bio.io import read_raman_file
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
models = ['anton532','anton785','kaiser','metrohm','mettler','tec','timegate','tornado']

raw_by_model =  {model: read_raman_file(name=model,level='raw') for model in models}

In [30]:
def get_wavenumbers(df:pd.DataFrame):
    cols_to_numeric = pd.to_numeric(pd.Series(np.array(df.columns)),errors='coerce')
    return cols_to_numeric[cols_to_numeric.notna()].to_numpy()

wavenumbers = {}
summary_stats={}

for model,df in raw_by_model.items():

    numeric_cols = pd.to_numeric(df.columns, errors="coerce") # type: ignore
    spectral_cols = df.columns[numeric_cols.notna()]
    model_wavenumbers = numeric_cols[numeric_cols.notna()].astype(float)

    spacing = np.diff(model_wavenumbers)

    wavenumbers[model] =  model_wavenumbers.to_numpy()
    summary_stats[model] = {
            'measurement_count': df.shape[0],
            'max_intensity': np.max(df[spectral_cols]),
            'min_intensity': np.min(df[spectral_cols]),
            'mean_intensity': np.mean(df[spectral_cols]),
            'min_wavenumber': model_wavenumbers.min(),
            'max_wavenumber': model_wavenumbers.max(),
            'wavenumber_counts': len(model_wavenumbers),
            "min_wn_spacing": spacing.min(),
            "max_wn_spacing": spacing.max(),
            "median_wn_spacing": np.median(spacing),
            "unique_spacing": np.unique(np.round(spacing, 4)),
            "roughly_evenly_spaced": np.allclose(spacing, np.median(spacing), atol=10**-4),
            'Memory Usage (Mb)': round(df.memory_usage().sum()*10**-6,2)
            }

summary_stats_df = pd.DataFrame(summary_stats).transpose()
summary_stats_df

,measurement_count,max_intensity,min_intensity,mean_intensity,min_wavenumber,max_wavenumber,wavenumber_counts,min_wn_spacing,max_wn_spacing,median_wn_spacing,unique_spacing,roughly_evenly_spaced,Memory Usage (Mb)
anton532,270,12267.88,1009.39,2197.199728,200.0,3500.0,1651,2.0,2.0,2.0,[2.0],True,3.58
anton785,270,6985.63,443.91,844.658705,100.0,2300.0,1101,2.0,2.0,2.0,[2.0],True,2.39
kaiser,134,11191.9518,2.5966,2400.279402,-36.3,1941.3001,6593,0.3,0.3001,0.3,"[0.3, 0.3001]",True,7.07
metrohm,399,25064.0,1057.0,2173.048186,202.22,3349.39,1917,1.1,2.37,1.6,"[1.1, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17...",False,6.14
mettler,275,54678.70571,99.111623,5268.838992,300.0,3200.0,2901,1.0,1.0,1.0,[1.0],True,6.39
tec,395,43539.529994,0.0,5092.860207,85.0,3210.0,3126,1.0,1.0,1.0,[1.0],True,9.89
timegate,133,0.005121,0.000043,0.000568,200.925569,1997.690789,511,1.918722,7.138847,3.497699,"[1.9187, 2.8324, 2.8349, 2.8373, 2.8397, 2.842...",False,0.55
tornado,385,91506.351462,251.38228,10174.176162,300.0,3300.0,3001,1.0,1.0,1.0,[1.0],True,9.26


In [31]:
print('** Wavenumber Ranges **')
name_width = max(len(model) for model in summary_stats)
for model,stats in summary_stats.items():
    print(f'{model:<{name_width}}: {stats['min_wavenumber']:>8.2f} - {stats['max_wavenumber']:>7.2f}')

** Wavenumber Ranges **
anton532:   200.00 - 3500.00
anton785:   100.00 - 2300.00
kaiser  :   -36.30 - 1941.30
metrohm :   202.22 - 3349.39
mettler :   300.00 - 3200.00
tec     :    85.00 - 3210.00
timegate:   200.93 - 1997.69
tornado :   300.00 - 3300.00


In [32]:
raw_transfer_df = read_raman_file(name='transfer_plate',level='raw')

raw_transfer_df[raw_transfer_df.columns[0]] == raw_transfer_df[raw_transfer_df.columns[-4]]
raw_transfer_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 2043,Unnamed: 2044,Unnamed: 2045,Unnamed: 2046,Unnamed: 2047,Unnamed: 2048,Analyte concentration,Glucose (g/L),Sodium Acetate (g/L),Magnesium Acetate (g/L)
0,sample1,[6293,7095,8325,9934,11917,14394,18925,34874,65535,...,1616,1024,1013,1067,1277,5618],sample1,4.619282,1.937172,1.052928
1,NaN,[6505,7332,8482,10175,12132,14792,19594,35813,65535,...,1655,1004,1032,1049,1271,5756],sample2,5.782718,1.175902,1.214738
2,sample2,[6478,7158,8444,9979,11932,14503,19309,35118,65535,...,1651,1024,1009,1049,1275,5685],sample3,3.953448,1.350473,2.132459
3,NaN,[6511,7308,8520,10205,12260,14777,19569,35825,65535,...,1623,1021,1008,1026,1250,5839],sample4,2.038084,0.948045,1.380240
4,sample3,[6561,7342,8562,10166,12202,14838,19593,35869,65535,...,1638,1010,1012,1047,1307,5801],sample5,4.978295,0.459765,2.539622


In [33]:
def clean_transfer_data(df: pd.DataFrame):

    df_copy = df.copy()
    
    df_copy[df_copy.columns[1]]=df_copy[df_copy.columns[1]].str.replace('[','')
    df_copy[df_copy.columns[-5]]=df_copy[df_copy.columns[-5]].str.replace(']','')
    feature_cols = ['sample'] + np.linspace(65,3350,2048).round(2).tolist()
    label_cols = df_copy.columns[-4:].tolist()
    df_copy.columns = feature_cols + label_cols
    df_copy['sample'] = df_copy['sample'].str.strip().ffill(axis=0)

    features = df_copy[feature_cols]
    labels = df_copy[label_cols]

    return features,labels

transfer_features_df, transfer_labels_df = clean_transfer_data(raw_transfer_df)
transfer_labels_df = transfer_labels_df[transfer_labels_df.notna().all(axis=1)]

clean_transfer_df = pd.merge(left = transfer_features_df, right=transfer_labels_df, left_on='sample',right_on='Analyte concentration',how='inner',copy=False).drop('Analyte concentration',axis=1)

clean_transfer_df
# plt.plot(np.linspace(65,3350,2048)[200:1100],values_0[200:1100])
# plt.yscale('linear')

/var/folders/dq/6g5fch396qb663c0xjt8xsxr0000gp/T/ipykernel_8002/969634548.py:20: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  clean_transfer_df = pd.merge(left = transfer_features_df, right=transfer_labels_df, left_on='sample',right_on='Analyte concentration',how='inner',copy=False).drop('Analyte concentration',axis=1)


,sample,65.0,66.6,68.21,69.81,71.42,73.02,74.63,76.23,77.84,...,3340.37,3341.98,3343.58,3345.19,3346.79,3348.4,3350.0,Glucose (g/L),Sodium Acetate (g/L),Magnesium Acetate (g/L)
0,sample1,6293,7095,8325,9934,11917,14394,18925,34874,65535,...,1603,1616,1024,1013,1067,1277,5618,4.619282,1.937172,1.052928
1,sample1,6505,7332,8482,10175,12132,14792,19594,35813,65535,...,1626,1655,1004,1032,1049,1271,5756,4.619282,1.937172,1.052928
2,sample2,6478,7158,8444,9979,11932,14503,19309,35118,65535,...,1615,1651,1024,1009,1049,1275,5685,5.782718,1.175902,1.214738
3,sample2,6511,7308,8520,10205,12260,14777,19569,35825,65535,...,1618,1623,1021,1008,1026,1250,5839,5.782718,1.175902,1.214738
4,sample3,6561,7342,8562,10166,12202,14838,19593,35869,65535,...,1601,1638,1010,1012,1047,1307,5801,3.953448,1.350473,2.132459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,sample94,6652,7453,8641,10270,12168,15014,20000,36732,65535,...,1620,1619,1026,1025,1078,1294,5834,6.531089,1.762051,1.429380
188,sample95,6798,7514,8786,10431,12372,15419,20547,37854,65535,...,1649,1587,1035,1033,1063,1271,5985,3.369372,1.400029,2.647925
189,sample95,6764,7534,8828,10532,12454,15504,20566,37705,65535,...,1632,1640,1030,1006,1073,1280,5925,3.369372,1.400029,2.647925
190,sample96,6847,7545,8795,10452,12588,15515,20492,37710,65535,...,1655,1667,1052,1019,1048,1266,6056,5.893708,1.898131,1.406329
